# 14A Appendix: Potential Outcomes, Selection Bias, and Adjustment


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/14A-Appendix-Causal-Inference-Details.ipynb)

This appendix holds the bookkeeping behind [Lecture 14](14-Causal-Inference.qmd): the six-student table that shows why a naive comparison can get the sign of an effect wrong, the decomposition of that comparison into a causal part plus **selection bias**, the formal statement of the backdoor criterion with its five-step procedure and a worked exercise, and the catalogue of estimators and natural-experiment designs that follow once an adjustment set is chosen. The lecture *states* each of these results; this is where they are worked out. The quiz asks you to define the ATE, read a Simpson's table, and apply the backdoor idea to a small graph, not to reproduce the derivations here.

For the estimators themselves run on simulated data (regression, stratification, inverse-propensity weighting, overlap diagnostics, difference-in-differences, instruments and regression discontinuity in detail), see the optional module [M14 — Causal Inference: Estimating Effects](M14-Causal-Inference-Estimation.qmd).

# Potential Outcomes in Detail

## If we could see everything (we can't)

The lecture's running example is an optional **tutoring program**: does it raise exam scores, when students choose whether to sign up? For each student $i$ there is a treatment $T_i \in \{0, 1\}$ and two potential outcomes, $Y_i(1)$ if tutored and $Y_i(0)$ if not, with individual effect $\tau_i = Y_i(1) - Y_i(0)$.

Imagine an all-seeing observer who knows **both** potential outcomes for six students. The three weaker students signed up for tutoring; the three stronger ones did not.

| Student | $T_i$ | $Y_i(0)$ | $Y_i(1)$ | $\tau_i$ |
|:-------:|:-----:|:--------:|:--------:|:--------:|
| Ann  | 1 | 60 | 62 | 2 |
| Bob  | 1 | 65 | 67 | 2 |
| Cara | 1 | 70 | 72 | 2 |
| Dan  | 0 | 68 | 70 | 2 |
| Eve  | 0 | 70 | 72 | 2 |
| Finn | 0 | 72 | 74 | 2 |

* Tutoring raises **every** student's score by exactly 2 points, so the average effect is 2. There is no ambiguity at all.
* Yet compare the scores we would *actually observe*: tutored students show their tutored scores, averaging $(62 + 67 + 72)/3 = 67$; untutored students show their untutored scores, averaging $(68 + 70 + 72)/3 = 70$. The naive comparison says tutoring **lowers** scores by 3.
* The gap is in the $Y_i(0)$ column: those who chose tutoring would have scored $65$ on average without it, versus $70$ for those who didn't. That $-5$ is **not** an effect of tutoring. It is who signed up. And $-5$ plus the real effect of $2$ gives exactly the $-3$ we saw.

## The fundamental problem of causal inference

Nobody is all-seeing. We only ever observe the outcome for the treatment actually received:

$$
Y_i^{\text{obs}} = T_i\, Y_i(1) + (1 - T_i)\, Y_i(0).
$$

The same table, as the world shows it to us:

| Student | $T_i$ | $Y_i(0)$ | $Y_i(1)$ | $\tau_i$ |
|:-------:|:-----:|:--------:|:--------:|:--------:|
| Ann  | 1 | [?]{style="color:#b0b0b0"} | 62 | [?]{style="color:#b0b0b0"} |
| Bob  | 1 | [?]{style="color:#b0b0b0"} | 67 | [?]{style="color:#b0b0b0"} |
| Cara | 1 | [?]{style="color:#b0b0b0"} | 72 | [?]{style="color:#b0b0b0"} |
| Dan  | 0 | 68 | [?]{style="color:#b0b0b0"} | [?]{style="color:#b0b0b0"} |
| Eve  | 0 | 70 | [?]{style="color:#b0b0b0"} | [?]{style="color:#b0b0b0"} |
| Finn | 0 | 72 | [?]{style="color:#b0b0b0"} | [?]{style="color:#b0b0b0"} |

The other potential outcome, the **counterfactual**, is missing, and every $\tau_i$ needs a grey cell. We can never compute $\tau_i$ for a single individual: causal inference is fundamentally a **missing data problem** [@imbens2015causal], and half the table is missing *not at random*. (The equation also quietly assumes each student's score depends only on their *own* treatment, not on anyone else's; this is the "stable unit treatment value" assumption, SUTVA.)

## ATE and ATT

Since individual effects are unknowable, we target population averages.

**Definitions**

The **Average Treatment Effect** is the expected difference between a unit's two potential outcomes, averaged over the whole population:

$$
\text{ATE} = \mathbb{E}[\,Y(1) - Y(0)\,] = \mathbb{E}[Y(1)] - \mathbb{E}[Y(0)].
$$

The **Average Treatment effect on the Treated** averages only over the units that actually received treatment:

$$
\text{ATT} = \mathbb{E}[\,Y(1) - Y(0) \mid T = 1\,].
$$


The second form of the ATE matters: we do not need both outcomes for the same person, only a good estimate of each average. On the tutoring table $\text{ATE} = \text{ATT} = 2$, because every $\tau_i$ is 2; they differ when the effect varies across units *and* the treated are not a random sample of everyone. The ATT answers "did the program help the people we actually enrolled?", the question a program manager is asked; the ATE answers "what would happen if we rolled it out to everyone?".

## Why we can't just compare treated vs. untreated

The tempting estimator is the **naive difference in means**, which is what we can measure:

$$
\mathbb{E}[Y \mid T=1] - \mathbb{E}[Y \mid T=0] = \mathbb{E}[Y(1)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0].
$$

Add and subtract the missing counterfactual $\mathbb{E}[Y(0)\mid T=1]$ (what the tutored students would have scored without tutoring) and regroup:

$$
\underbrace{\mathbb{E}[Y(1)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0]}_{\text{naive difference}}
= \underbrace{\mathbb{E}[Y(1)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}1]}_{\text{ATT}}
+ \underbrace{\mathbb{E}[Y(0)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0]}_{\text{selection bias}}.
$$

Check it against the all-seeing table: $\text{ATT} = 2$, selection bias $= 65 - 70 = -5$, and $2 + (-5) = -3$, exactly the naive difference. The arithmetic was right; the causal reading was wrong.

## Selection bias

**Definition**

**Selection bias** is the difference in outcomes that treated and untreated units would have shown *even without any treatment*, $\mathbb{E}[Y(0)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0]$. It is nonzero whenever *who gets treated* depends on things that also affect the outcome.


* It is exactly the hospital example from the lecture: patients who visit are sicker to begin with. And exactly the tutoring table: the students who signed up were weaker to begin with.
* The coefficient on `T` in the regression `Y ~ T` *is* the naive difference in means, so it inherits the same bias. A regression table does not launder it away. Adding the measured confounders to the formula, `Y ~ T + Z`, is the same move as comparing within strata of $Z$; it reduces the bias only to the extent that $Z$ captures what drove selection.

**A note on terminology**

We use *selection bias* in the sense above, a baseline difference between those who were and were not treated (the econometrics usage [@angrist2009mostly]). Epidemiologists reserve the term for bias caused by how units get *into the sample*. The lecture calls that **collider bias** and treats it as a separate mechanism: selection bias in our sense comes from a fork, collider bias from a collider. Keep the two apart.


## Why randomization removes selection bias

If a coin flip decides $T_i$, treatment is statistically independent of the potential outcomes,

$$
\{Y_i(0), Y_i(1)\} \perp\!\!\!\perp T_i ,
$$

and independence does two things at once.

1. **It zeroes the selection-bias term.** Conditioning on $T$ no longer changes the distribution of $Y(0)$, so $\mathbb{E}[Y(0)\mid T{=}1] = \mathbb{E}[Y(0)\mid T{=}0]$. In expectation the two groups are identical in every respect, measured *and* unmeasured, except the treatment. No list of confounders is needed.
2. **It makes the treated a representative sample**, so $\mathbb{E}[Y(1) - Y(0) \mid T{=}1] = \mathbb{E}[Y(1) - Y(0)]$, i.e. $\text{ATT} = \text{ATE}$.

Put the two together with the decomposition above: the naive difference equals the ATT plus zero, and the ATT equals the ATE, so the naive difference in means is an **unbiased estimate of the ATE** [@imbens2015causal]. That is the whole case for randomized experiments, A/B tests and clinical trials included.

Try it in the notebook of Lecture 14: in the randomization simulation, change `p_treat_obs` so that *healthier* patients are more likely to be treated, and predict the direction of the bias before you run the cell.

# The Backdoor Criterion in Detail

## The formal statement

A **backdoor path** is a non-causal path from $X$ to $Y$ that starts with an arrow *into* $X$ (e.g. $X \leftarrow Z \to Y$). Backdoor paths are the graph-language for confounding.

**Backdoor criterion**

A set of variables $S$ suffices to identify the causal effect of $X$ on $Y$ if $S$ **blocks every backdoor path** from $X$ to $Y$ and contains **no descendants of $X$** [@pearl2009causality].


When such an $S$ exists, the **direct standardization** the lecture used on the kidney stones applies with $S$ in place of the stratifying variable:

$$
P(Y \mid do(X{=}x)) = \sum_{s} P(Y \mid X{=}x, S{=}s)\, P(S{=}s).
$$

The backdoor criterion is what tells you the formula is *legitimate*. It was legitimate for kidney-stone size because stone size is a fork. In potential-outcomes language a valid $S$ gives **conditional ignorability**,

$$
\{Y(0), Y(1)\} \perp\!\!\!\perp T \mid S ,
$$

randomization's independence, now holding only *within* levels of $S$. The DAG is how we argue it; it cannot be checked from data.

## The backdoor procedure, by hand

Given a DAG, a treatment $X$ and an outcome $Y$:

1. **List every path** from $X$ to $Y$ (ignore arrow direction while tracing; visit no node twice).
2. **Set aside the causal paths**, those where every arrow points away from $X$ toward $Y$. Never block these (that would be adjusting for a mediator).
3. **Classify the rest.** A path whose first arrow points *into* $X$ is a **backdoor path**: mark it open or blocked (forks and chains open, colliders closed). A path whose first arrow points *out of* $X$ but that is not causal, because it runs into a collider as in $X \to C \leftarrow Y$, is **neither**: blocked by default, and it needs no action *as long as you leave $C$ and its descendants alone*. People lose points by calling that a backdoor path, so watch for it.
4. **Choose $S$** that blocks every open backdoor path, a fork or chain node on each, containing **no collider or collider-descendant on *any* path** (backdoor or not) and **no descendant of $X$**.
5. **Report $S$.** There may be several valid choices; the smallest is usually best, since every extra variable is another chance to have drawn an arrow wrong.

Then adjust for $S$ by whichever estimator you like; the next section lists them.

## Your turn: pick the adjustment set

Tutoring ($X$) and final grade ($Y$), with
$\text{GPA}_{\text{prior}} \to X$, $\text{GPA}_{\text{prior}} \to Y$,
$X \to \text{Hours} \to Y$, and $X \to \text{HonorRoll} \leftarrow Y$.

Which of these adjustment sets are valid for the **total** effect of $X$ on $Y$?

* (a) $\{\text{GPA}_{\text{prior}}\}$
* (b) $\{\text{GPA}_{\text{prior}}, \text{Hours}\}$
* (c) $\{\text{GPA}_{\text{prior}}, \text{HonorRoll}\}$
* (d) $\{\}$ (nothing)

Run the five steps before you read on.

## Your turn: the answer

In [ ]:
# Setup for Colab and for a local clone of the Materials repo: make the course's
# helper modules importable, downloading any that are not already available.
import importlib.util, os, sys, urllib.request

_BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main"
for _path in ["scripts/slideUtilities.py"]:
    _name = os.path.splitext(os.path.basename(_path))[0]
    if importlib.util.find_spec(_name) is not None:
        continue
    for _root in (".", ".."):                      # a clone: notebook in jupyter_notebooks/
        if os.path.exists(os.path.join(_root, _path)):
            sys.path.insert(0, os.path.abspath(os.path.dirname(os.path.join(_root, _path))))
            break
    else:                                          # Colab: fetch the file
        os.makedirs("modules", exist_ok=True)
        urllib.request.urlretrieve(f"{_BASE}/{_path}", os.path.join("modules", os.path.basename(_path)))
        if os.path.abspath("modules") not in sys.path:
            sys.path.insert(0, os.path.abspath("modules"))


In [ ]:
#| echo: false
#| fig-align: center
#| label: tutoring-dag
import networkx as nx
import matplotlib.pyplot as plt
from slideUtilities import OKABE_ITO

def draw(ax, edges, pos, title="", node_color="#cde", node_size=1400, font_size=12):
    """Lecture-14 DAG style: pale nodes, dark outlines, black arrows."""
    G = nx.DiGraph(edges)
    colors = [node_color.get(v, "#cde") for v in G] if isinstance(node_color, dict) else node_color
    nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors,
            node_size=node_size, font_size=font_size, arrowsize=22,
            edgecolors="#333", linewidths=1.2)
    ax.set_title(title, fontsize=11)

tutor_edges = [("GPA\nprior", "X"), ("GPA\nprior", "Y"), ("X", "Hours"), ("Hours", "Y"),
               ("X", "Honor\nroll"), ("Y", "Honor\nroll")]
tutor_pos = {"X": (0, 0), "GPA\nprior": (1, 1.25), "Hours": (1, 0.3),
             "Honor\nroll": (1, -0.95), "Y": (2, 0)}
roles = {"GPA\nprior": OKABE_ITO[2], "Hours": OKABE_ITO[0], "Honor\nroll": OKABE_ITO[3]}

fig, ax = plt.subplots(figsize=(4.6, 3.2))
draw(ax, tutor_edges, tutor_pos, node_color=roles, node_size=1700, font_size=9,
     title="X = tutoring, Y = final grade\n"
           "green = confounder, orange = mediator, red = collider")
ax.set_xlim(-0.3, 2.3); ax.set_ylim(-1.35, 1.6)
plt.tight_layout(); plt.show()

| Path | Type | Action |
|:--|:--|:--|
| $X \to \text{Hours} \to Y$ | causal (open) | leave alone |
| $X \leftarrow \text{GPA} \to Y$ | backdoor (open fork) | block with GPA |
| $X \to \text{Honor} \leftarrow Y$ | **neither** (blocked collider) | do not touch |

: {tbl-colwidths="[44,34,22]"}

**Only (a).** (b) blocks the mediator, so you lose the part of the effect that works through study hours. (c) opens the collider. (d) leaves the backdoor through GPA open. Prior GPA was measured before anyone signed up, so it is not a descendant of tutoring, and it is not a collider anywhere. And "let's restrict the study to honor-roll students, it's a cleaner sample" is (c) in disguise: restricting a sample *is* conditioning.

# Estimating Effects

## Four ways to adjust for $S$

Once $S$ is chosen, any of these recovers the effect:

1. **Regression adjustment**: include $S$ as covariates and read off the coefficient on $T$. This assumes the linear form is right; the regression is an estimator, not a licence.
2. **Stratification / standardization**: estimate the effect within levels of $S$, then average with the **population's** stratum shares (giving the ATE); use the *treated* group's shares and you get the ATT.
3. **Matching**: pair each treated unit with similar control unit(s) on $S$.
4. **Propensity scores**: match or weight on $e(S) = P(T{=}1 \mid S)$, a one-number summary of $S$.

Identification (which $S$) and estimation (which estimator) are separate decisions. A *wrong* $S$ gives a wrong answer with a tight standard error; no estimator rescues a bad graph.

## Propensity scores

When $S$ is high-dimensional, matching directly is hard. Rosenbaum and Rubin showed it is enough to match or weight on the scalar **propensity score** $e(S) = P(T{=}1\mid S)$ [@rosenbaum1983central].

* **Estimate it** with any classifier of $T$ from $S$; logistic regression (Lecture 11A) is the usual choice.
* **Match** each treated unit to controls with a similar $e$, or **weight** each unit by $1/e(S)$ if treated and $1/(1 - e(S))$ if not (inverse-propensity weighting, IPW).
* Weighting builds a "pseudo-population" in which treatment is unrelated to $S$, mimicking a randomized experiment from observational data.

**Check overlap (positivity) first**

Plot the histogram of $e$ for treated and for control units. Where one group has units and the other has none, there is no comparable counterpart and the estimate is extrapolation. An $e$ of $0.01$ gives a weight of $100$: a tight standard error around a number that means nothing.


## When confounders are *unobserved*: natural experiments

Adjustment only works for confounders we can **measure**. When we can't, we look for "nature's randomization", sources of variation in treatment that are *as good as random* [@angrist2014mastering].

* **Difference-in-differences (DiD)**: compare the *change* over time in a treated group with the change in a control group. The groups need not be alike, only move alike: the assumption is **parallel trends**.
* **Regression discontinuity (RD)**: treatment switches on at a cutoff; compare units just above and just below it. The assumption is that **nothing else jumps at the cutoff**.
* **Instrumental variables (IV)**: a variable that nudges the treatment as good as at random, easiest to see as a graph.

## An instrument, in graph terms

The classic instrument is the Vietnam draft lottery $L$ for military service $T$, with earnings $Y$ as the outcome.

In [ ]:
#| echo: false
#| fig-align: center
#| label: iv-dag
iv_pos = {"L\nlottery": (0, 0), "T": (1, 0), "Y": (2, 0), "U\nunobserved": (1.5, 1)}
fig, ax = plt.subplots(figsize=(6.4, 2.8))
draw(ax, [("L\nlottery", "T"), ("T", "Y")], iv_pos, node_size=2000, font_size=9)
U = nx.DiGraph([("U\nunobserved", "T"), ("U\nunobserved", "Y")])
nx.draw_networkx_nodes(U, iv_pos, ax=ax, nodelist=["U\nunobserved"], node_color="white",
                       node_size=2900, edgecolors="#999", linewidths=1.2)
nx.draw_networkx_labels(U, iv_pos, ax=ax, labels={"U\nunobserved": "U\nunobserved"},
                        font_size=9, font_color="#777")
nx.draw_networkx_edges(U, iv_pos, ax=ax, node_size=2900, style="dashed",
                       edge_color="#999", arrowsize=22)
nx.draw_networkx_edges(nx.DiGraph([("L\nlottery", "Y")]), iv_pos, ax=ax, node_size=2000,
                       style="dashed", edge_color="#c0392b", width=2, arrowsize=22,
                       connectionstyle="arc3,rad=0.3")
ax.text(1, -0.55, "✗", color="#c0392b", fontsize=22, ha="center", va="center",
        bbox=dict(boxstyle="circle,pad=0.1", fc="white", ec="none"))
ax.text(1, -0.8, "no direct L → Y (exclusion)", color="#c0392b", fontsize=9,
        ha="center", va="top")
ax.set_xlim(-0.4, 2.4); ax.set_ylim(-1.1, 1.35)
plt.tight_layout(); plt.show()

1. **Relevance**: the arrow $L \to T$ exists (the lottery really changes who serves).
2. **Exclusion**: no arrow $L \to Y$; the lottery affects earnings *only through* service.
3. **Independence**: no backdoor path from $L$ to $Y$; nothing points into $L$ (it is random).

The unobserved $U$ confounds $T$ and $Y$, so adjustment is impossible, but $L$ is untouched by $U$, and that is all IV needs: any difference in earnings between low and high lottery numbers can only have come through service.

## Assumptions you can't skip

Every observational causal estimate relies on assumptions [@hernan2020whatif]:

* **Conditional ignorability**: $\{Y(0),Y(1)\} \perp\!\!\!\perp T \mid S$, i.e. the adjustment set blocks every backdoor path. The DAG argues it; data cannot check it.
* **Positivity (overlap)**: every unit has a nonzero chance of each treatment, $0 < e(S) < 1$.
* **Correct graph**: the DAG reflects reality.

Since ignorability is untestable, good practice includes a **sensitivity analysis**: how strong would an unmeasured confounder have to be to overturn the conclusion?

## Going further

The optional module [M14 — Causal Inference: Estimating Effects](M14-Causal-Inference-Estimation.qmd) runs each estimator above on simulated data (regression, stratification, IPW with an overlap check), shows that a valid adjustment set gives the same answer whichever estimator you use, works DiD, IV and RD in detail, and ends with practice scenarios. Software for the draw → identify → estimate → refute workflow: [DoWhy](https://www.pywhy.org/dowhy/) and [dagitty.net](https://dagitty.net/).

## Bibliography
